In [1]:
import pickle
import onnxmltools
from onnxmltools.convert.common.data_types import FloatTensorType
import numpy as np
import skl2onnx
from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType as SklearnFloatType

for name in ["lgb", "xgb"]:
    try:
        print(f"Обработка {name}...")
        
        with open(f"DATA/real_time/{name}_final_model.pkl", "rb") as f:
            clf_model = pickle.load(f)
        
        with open(f"DATA/real_time/hybrid_pipeline_{name}.pkl", "rb") as f:
            hybrid = pickle.load(f)
        
        reg_model = hybrid["reg_model"]
        
        # ================= CLASSIFIER =================
        if name == "lgb":
            onnx_clf = onnxmltools.convert_lightgbm(
                clf_model, initial_types=[("input", FloatTensorType([None, 50]))]
            )
        elif name == "xgb":
            onnx_clf = onnxmltools.convert_xgboost(
                clf_model, initial_types=[("input", FloatTensorType([None, 50]))]
            )
        
        with open(f"clf_{name}.onnx", "wb") as f:
            f.write(onnx_clf.SerializeToString())
        print(f"  Классификатор сохранён как clf_{name}.onnx")
        
        # ================= REGRESSOR =================
        if name == "lgb":
            onnx_reg = onnxmltools.convert_lightgbm(
                reg_model, initial_types=[("input", FloatTensorType([None, 50]))]
            )
        elif name == "xgb":
            onnx_reg = onnxmltools.convert_xgboost(
                reg_model, initial_types=[("input", FloatTensorType([None, 50]))]
            )
        
        with open(f"reg_{name}.onnx", "wb") as f:
            f.write(onnx_reg.SerializeToString())
        print(f"  Регрессор сохранён как reg_{name}.onnx")
        
    except FileNotFoundError as e:
        print(f"  Файл не найден для {name}: {e}")
    except Exception as e:
        print(f"  Ошибка при обработке {name}: {e}")

print("\nONNX экспорт готов")

Обработка lgb...
  Классификатор сохранён как clf_lgb.onnx
  Регрессор сохранён как reg_lgb.onnx
Обработка xgb...
  Классификатор сохранён как clf_xgb.onnx
  Регрессор сохранён как reg_xgb.onnx

ONNX экспорт готов
